# Example 2: Six-Frame ORF Finding — Locating the SARS-CoV-2 Spike Gene

Fetches a padded window (genome positions 21,400–25,450, not the gene's exact published
coordinates) of the SARS-CoV-2 reference genome (`NC_045512.2`) around the spike (S) gene,
and scans all six reading frames (3 forward + 3 reverse) for open reading frames, so the
scan has to *rediscover* the ORF rather than being handed its location.


**Expected result:** the standout ORF should be ~1282 aa in forward frame +2 — the
spike gene (coordinates 21,563–25,384), matching the known spike glycoprotein length.

In [ ]:
# Fetch a padded window around the spike gene and scan all six reading frames
import os
from Bio import Entrez, SeqIO

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)

Entrez.email = "you@example.com"
# seq_start/seq_stop fetch only this sub-region, not the full 29,903 bp genome
handle = Entrez.efetch(db="nucleotide", id="NC_045512.2", rettype="fasta", retmode="text",
                        seq_start=21400, seq_stop=25450)
record = SeqIO.read(handle, "fasta")
handle.close()
SeqIO.write(record, "data/NC_045512.2_spike_region.fasta", "fasta")
print(f"Fetched region length: {len(record.seq)} bp")

def find_orfs(seq, min_aa=100):
    """Scan one strand's three reading frames for ORFs at least min_aa long."""
    orfs = []
    for frame in range(3):
        trans = str(seq[frame:].translate(table=1, to_stop=False))
        start = None
        for i, aa in enumerate(trans):
            if aa == "M" and start is None:
                start = i
            if aa == "*" and start is not None:
                length = i - start
                if length >= min_aa:
                    nt_start = frame + start * 3
                    nt_end = frame + (i + 1) * 3
                    orfs.append((frame, nt_start, nt_end, length))
                start = None
    return orfs

forward_orfs = find_orfs(record.seq)
revcomp = record.seq.reverse_complement()
reverse_orfs = find_orfs(revcomp)

print(f"\nForward-strand ORFs >= 100 aa: {len(forward_orfs)}")
for frame, s, e, length in sorted(forward_orfs, key=lambda x: -x[3]):
    print(f"  frame +{frame+1}: {s}-{e} nt, {length} aa")

print(f"\nReverse-strand ORFs >= 100 aa: {len(reverse_orfs)}")
for frame, s, e, length in sorted(reverse_orfs, key=lambda x: -x[3]):
    print(f"  frame -{frame+1}: {s}-{e} nt, {length} aa")

longest = max(forward_orfs + reverse_orfs, key=lambda x: x[3])
strand = "forward" if longest in forward_orfs else "reverse"
print(f"\nLongest ORF overall: {longest[3]} aa, frame {longest[0]+1} ({strand} strand)")

with open("results/example2_spike_orfs.csv", "w") as f:
    f.write("strand,frame,nt_start,nt_end,aa_length\n")
    for frame, s, e, length in forward_orfs:
        f.write(f"forward,{frame+1},{s},{e},{length}\n")
    for frame, s, e, length in reverse_orfs:
        f.write(f"reverse,{frame+1},{s},{e},{length}\n")
print("\nSaved: data/NC_045512.2_spike_region.fasta, results/example2_spike_orfs.csv")

Fetched region length: 4051 bp

Forward-strand ORFs >= 100 aa: 1
  frame +2: 136-3985 nt, 1282 aa

Reverse-strand ORFs >= 100 aa: 0

Longest ORF overall: 1282 aa, frame 2 (forward strand)

Saved: data/NC_045512.2_spike_region.fasta, results/example2_spike_orfs.csv


/home/moh/miniforge3/envs/bioinfo-skill3-translation-orf/lib/python3.11/site-packages/Bio/Seq.py:2880: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(
